# 10-2 元組進階操作與可雜湊特性（Hashable）

- **單元編號**：10-2
- **學習目標**：
  1. 深入探究可變（Mutable）與不可變（Immutable）物件在記憶體中的運作本質與 `id()` 追蹤。
  2. 理解「可雜湊性（Hashability）」核心概念，洞悉為什麼只有不可變物件能作為雜湊鍵值。
  3. 掌握巢狀元組（Nested Tuples）與結構化多欄位資料的層級封裝與多層存取。
  4. 實作元組與方向向量的座標位移運算，建立幾何網格位移模型。
  5. 掌握多傳回值模擬機制，為後續模組化架構提早奠定穩固基礎。
  6. 精通元組比較大小（Tuple Comparison）與字典序規則，解鎖多條件排序的核心原型。
- **適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者
- **先備知識**：10-1 元組基本建立與不可變性、第 9 章二維方向向量概念

---


### 10.2.1 可變（Mutable）與不可變（Immutable）物件的記憶體行為本質

在上一單元中，我們感受到元組「無法原地修改」的特性。但你有沒有想過：在電腦的記憶體底層，到底發生了什麼事？為什麼串列可以隨時改，而整數、浮點數、字串與元組卻不行？

#### 🔍 照妖鏡：內建函數 `id()`
在 Python 中，每一個物件在記憶體中都有一個獨一無二的門牌號碼，我們可以使用內建函數 **`id(obj)`** 查看它的記憶體位址！
- **可變物件（Mutable，如 List 串列）**：
  當我們執行 `my_list.append(99)` 或 `my_list[0] = 5` 時，`id(my_list)` **完全不會改變**！就像同一個白板，上面的字雖然換了，但白板本身還是掛在原來那間教室的同一個掛鉤上。
- **不可變物件（Immutable，如 int, str, tuple）**：
  當我們自以為在「修改」一個數值或字串時（例如 `x = 10; x = x + 1`），如果你查驗 `id(x)`，會震驚地發現門牌號碼完全變了！Python 是在記憶體中**重新開闢了一間新房間放 11**，然後把名牌 `x` 撕下來貼到新房間去！

#### ⚠️ 特殊邊界：元組內含可變串列的現象
如果一個元組內部裝了一個串列，例如 `t = ([1, 2], 3)`：
元組保護的是「**它手中握著的房間門牌號碼**」。元組記住了「第 0 格是一間串列房間、第 1 格是整數 3」，這兩個門牌永遠不能換！但是，串列房間內部的人員進出（`t[0].append(99)`），元組是管不著的！
因此在標準的軟體開發與 APCS 考場中，我們強烈建議：**元組內部應該全部存放不可變資料（如整數、字串、其他元組），打造絕對純淨堅固的安全實體！**


In [ ]:
# 範例 10.2.1：使用 id() 觀察可變串列 vs 不可變元組的記憶體行為

# 1. 串列（可變）：原地修改，記憶體位址 id 保持不變
lst = [10, 20]
print("串列修改前 id：", id(lst))
lst.append(30)
lst[0] = 99
print("串列修改後 id：", id(lst), "（位址完全相同！）")

# 2. 元組（不可變）：任何修改嘗試都是產生全新物件
t = (10, 20)
print()
print("元組原始 id：", id(t))
# 透過串接看似「加上新元素」，實則是建立了全新元組
t = t + (30,)
print("串接後的新元組 id：", id(t), "（位址已變為全新空間！）")

# 3. 探測元組內含串列的特殊現象
hybrid = ([1, 2], "Fixed")
print()
print("複合元組原始內容：", hybrid)
# 修改內含的串列（合法的，因為串列門牌沒變）
hybrid[0].append(3)
print("修改內含串列後的內容：", hybrid)


In [ ]:
# 填空 10.2.1：使用 id() 驗證記憶體位址變化
# 請將 ___ 替換為正確的函數名稱 id

a = (100, 200)
original_address = ___(a)

# 重新賦值產生全新元組
a = (100, 200, 300)
new_address = ___(a)

is_same_object = (original_address == new_address)
print("重新賦值前後記憶體位址是否相同：", is_same_object)


In [ ]:
# 練習 10.2.1：不可變字串與元組位址驗證儀
# 題目說明：輸入一個整數 X。
# 建立一個單元素元組 `t1 = (X,)`，記錄其 id 為 `id1`。
# 接著執行 `t2 = t1 + (X * 2,)`，記錄其 id 為 `id2`。
# 輸出兩行：
# 第一行：t2 的元素內容（以空白隔開）。
# 第二行：判斷 id1 是否等於 id2？若相同輸出 "SAME"，若不同輸出 "DIFFERENT"。

# 【公開測試資料 1】
# 5
# 輸出：
# 5 10
# DIFFERENT

# 【公開測試資料 2】
# 100
# 輸出：
# 100 200
# DIFFERENT

# 請在此處撰寫你的程式碼：
x = int(input())

t1 = (x,)
id1 = id(t1)

t2 = t1 + (x * 2,)
id2 = id(t2)

print(*t2)
if id1 == id2:
    print("SAME")
else:
    print("DIFFERENT")


In [ ]:
# 挑戰 10.2.1：純淨元組檢測器
# 題目說明：給定一個容器列表。
# 檢驗該列表中所包含的各項元素是否「全為不可變的基本型態（int, float, str, tuple）」？
# 若全為不可變型態輸出 "SAFE"，若包含可變型態（如 list）輸出 "MUTABLE DETECTED"。
# 本題無公開測試資料，請自行測試。

# 請在此處撰寫你的程式碼：
test_items = [10, "Hello", (1, 2), 3.14]

is_safe = True
for item in test_items:
    if type(item) is list:
        is_safe = False
        break

if is_safe:
    print("SAFE")
else:
    print("MUTABLE DETECTED")


### 10.2.2 什麼是「可雜湊（Hashable）」？為什麼只有不可變物件能被雜湊？

我們常常聽到高階演算法中提到「**雜湊（Hash）**」、「雜湊表（Hash Table）」或是下一節即將學到的「字典（Dict）」與「集合（Set）」。這些以超光速 $O(1)$ 速度著稱的資料結構，背後仰賴的唯一基石就是——**物件必須是「可雜湊的（Hashable）」**！

#### 🔑 生活比喻：身分證指紋機
想像一間最高機密保險庫：
- 當你要存放一件物品時，管理員會把物品放到「**指紋掃描機（雜湊函數 Hash Function）**」上，計算出一串專屬的固定號碼（雜湊值 Hash Value），然後把物品存入該號碼對應的專屬儲物格中。
- 當你下次要找這件物品時，管理員不用從第 1 格翻到第 10000 格（$O(N)$ 線性搜尋），只要再掃描一次這件物品算出號碼，直接走向該格子，瞬間取出（$O(1)$ 查找）！

#### 💥 為什麼串列（List）絕對不能被雜湊？
現在想像一下：如果你把一個串列 `[1, 2]` 當作識別證放進儲物櫃。
結果過了一會兒，某人偷偷對這個串列執行了 `append(99)`，串列內容變成了 `[1, 2, 99]`！
當管理員再次掃描它時，算出來的號碼全變了，電腦根本找不到它原本被放在哪裡，整套系統立刻全面崩潰！
這就是為什麼 Python 規定：**只有內容一生一世絕不改變的不可變物件（如 int, str, tuple），才具備計算固定指紋的資格，稱為 Hashable！**

在 Python 中，你可以使用內建函數 **`hash(obj)`** 查驗物件的雜湊指紋：
- `hash((1, 2))` ➔ 算出正常整數指紋（合法！）
- `hash([1, 2])` ➔ 立即崩潰報錯：`TypeError: unhashable type: 'list'`（不可雜湊！）


In [ ]:
# 範例 10.2.2：使用 hash() 體驗可雜湊指紋與不可雜湊報錯

# 1. 不可變物件具備穩定的雜湊值（指紋）
num_hash = hash(42)
str_hash = hash("APCS")
tuple_hash = hash((3, 5))

print("整數 42 的雜湊值：", num_hash)
print("字串 'APCS' 的雜湊值：", str_hash)
print("座標元組 (3, 5) 的雜湊值：", tuple_hash)

# 2. 探測不可雜湊物件
# 若嘗試計算串列的 hash()，Python 會立即拋出 TypeError
# hash([3, 5])  # TypeError: unhashable type: 'list'

# 3. 使用 try ... except 安全捕捉不可雜湊錯誤
test_candidates = [(1, 2), [1, 2], "Hello", 999]

print()
print("--- 雜湊資格檢驗表 ---")
for candidate in test_candidates:
    try:
        h = hash(candidate)
        print(f"型態 {type(candidate).__name__:>5}：合格！雜湊值 = {h}")
    except TypeError:
        print(f"型態 {type(candidate).__name__:>5}：❌ 違規！Unhashable 不可雜湊！")


In [ ]:
# 填空 10.2.2：檢查候選資料是否具備雜湊能力
# 請將 ___ 替換為正確的內建函數名稱 hash

coord = (10, 20)

# 計算不可變元組的雜湊指紋
h_val = ___(coord)

print("座標元組成功生成雜湊指紋：", h_val)


In [ ]:
# 練習 10.2.2：元組雜湊一致性驗證
# 題目說明：輸入兩個整數 A, B。
# 分別建立兩個獨立宣告的元組：
# `t1 = (A, B)` 與 `t2 = (A, B)`。
# 請計算兩者的 hash 值：`h1 = hash(t1)` 與 `h2 = hash(t2)`。
# 輸出兩行：
# 第一行：印出 t1 的雜湊值是否等於 t2 的雜湊值（若是輸出 "EQUAL"，若否輸出 "NOT EQUAL"）。
# 第二行：輸出 "A, B 可以安全作為字典的鍵值" 標籤提示。

# 【公開測試資料 1】
# 7 9
# 輸出：
# EQUAL
# HASHABLE OK

# 【公開測試資料 2】
# -3 100
# 輸出：
# EQUAL
# HASHABLE OK

# 請在此處撰寫你的程式碼：
a, b = map(int, input().split())

t1 = (a, b)
t2 = (a, b)

h1 = hash(t1)
h2 = hash(t2)

if h1 == h2:
    print("EQUAL")
else:
    print("NOT EQUAL")

print("HASHABLE OK")


In [ ]:
# 挑戰 10.2.2：動態資料不可變性守護檢測
# 題目說明：輸入一行包含 N 個以空白隔開的整數。
# 請將其封裝為元組 T。
# 請印出該元組的雜湊值是否為整數（isinstance(hash(T), int)）？
# 若成功印出 "PASSED"，若過程發生錯誤印出 "FAILED"。
# 本題無公開測試資料，請自行測試。

# 請在此處撰寫你的程式碼：
raw_vals = tuple(map(int, input().split()))
try:
    h = hash(raw_vals)
    if isinstance(h, int):
        print("PASSED")
    else:
        print("FAILED")
except:
    print("FAILED")


### 10.2.3 巢狀元組（Nested Tuples）與結構化多欄位資料打包（如 `(id, name, score)`）

在真實世界的程式設計與競賽中，資料往往不是單純的一串數字，而是具有層級與結構關聯的複合實體。例如一位選手的資料可能包含：
- 學號（整數）
- 姓名（字串）
- 多項成績（國、英、數三科成績組成的子元組）

#### 🪆 像俄羅斯套娃一樣嵌套：巢狀元組
元組內部可以容納任何型態，當然也包括「**另一個元組**」！這種元組裡面包元組的結構稱為 **巢狀元組（Nested Tuples）**。
```python
student = (101, "Alice", (95, 88, 92))
```
當我們要提取巢狀元組深處的資料時，使用**多重中括號索引**：
- `student[0]` ➔ 抓出學號 `101`
- `student[1]` ➔ 抓出姓名 `"Alice"`
- `student[2]` ➔ 抓出整包成績子元組 `(95, 88, 92)`
- `student[2][0]` ➔ 抓出成績子元組的第 0 項國文成績 `95`！

#### 💼 結構化優勢：免類別物件的超輕量封裝
在還沒學習物件導向（Class）之前，巢狀元組是組織複雜多欄位資料最乾淨、最不耗費額外記憶體、且天生不可竄改的極致利器！


In [ ]:
# 範例 10.2.3：巢狀元組的結構封裝與多層存取

# 定義一個包含二維座標與點屬性的巢狀元組
# 格式：(點編號, (列座標, 行座標), 點名稱)
node = (1, (2, 4), "Checkpoint_A")

print("節點完整結構：", node)
print("節點編號：", node[0])
print("座標子元組：", node[1])
print("列座標 r：", node[1][0])
print("行座標 c：", node[1][1])
print("節點名稱：", node[2])

# 巢狀解包：可以直接拆解外層與內層！
node_id, (r, c), node_name = node
print()
print(f"一鍵解包成果：ID={node_id}, 位於第 {r} 列第 {c} 行，名稱為 {node_name}")


In [ ]:
# 填空 10.2.3：多層巢狀解包提取學生數學成績
# 請將 ___ 替換為正確的變數名稱

record = (101, "David", (88, 92, 95))

# 格式解包：學號, 姓名, (國, 英, 數)
sid, name, (chi, eng, ___) = record

print("姓名：", name)
print("數學成績：", math)


In [ ]:
# 練習 10.2.3：學生卡成績統計器
# 題目說明：輸入一行包含：
# 學號（整數）、姓名（字串）、三次段考成績（三個整數）。
# 請將其打包為巢狀元組：`(sid, name, (score1, score2, score3))`。
# 請透過巢狀解包或多層索引計算三次成績的總分與整數平均（// 3）。
# 輸出兩行：
# 第一行：學號與姓名（以空白隔開）
# 第二行：總分與平均分（以空白隔開）

# 【公開測試資料 1】
# 101 Amy 80 90 100
# 輸出：
# 101 Amy
# 270 90

# 【公開測試資料 2】
# 205 Ken 70 75 80
# 輸出：
# 205 Ken
# 225 75

# 請在此處撰寫你的程式碼：
parts = input().split()
sid = int(parts[0])
name = parts[1]
scores = (int(parts[2]), int(parts[3]), int(parts[4]))

student_card = (sid, name, scores)

# 解包統計
s_id, s_name, (s1, s2, s3) = student_card
total = s1 + s2 + s3
avg = total // 3

print(s_id, s_name)
print(total, avg)


In [ ]:
# 挑戰 10.2.3：二維多線段長度統計
# 題目說明：輸入兩條線段，每行輸入四個整數：r1, c1, r2, c2 代表線段兩端點。
# 將每條線段儲存為巢狀元組：`((r1, c1), (r2, c2))`。
# 計算兩條線段的曼哈頓長度總和：(|r1 - r2| + |c1 - c2|)。
# 輸出兩線段長度之和。
# 本題無公開測試資料，請自行測試。

# 請在此處撰寫你的程式碼：
line1_pts = list(map(int, input().split()))
line2_pts = list(map(int, input().split()))

seg1 = ((line1_pts[0], line1_pts[1]), (line1_pts[2], line1_pts[3]))
seg2 = ((line2_pts[0], line2_pts[1]), (line2_pts[2], line2_pts[3]))

len1 = abs(seg1[0][0] - seg1[1][0]) + abs(seg1[0][1] - seg1[1][1])
len2 = abs(seg2[0][0] - seg2[1][0]) + abs(seg2[0][1] - seg2[1][1])

print(len1 + len2)


### 10.2.4 元組在二維座標位移運算中的應用（`(r + dr, c + dc)`）

在第九章 9-7 節中，我們學會了使用方向向量差值陣列 `dr = [-1, 1, 0, 0]` 與 `dc = [0, 0, -1, 1]` 來探測相鄰的四個格子。當時我們用兩個平行的數值變數進行步進：
```python
nr = r + dr[d]
nc = c + dc[d]
```
現在結合了元組，我們可以將這套二維網格導航系統升級為更簡潔、更直觀的**向量元組模型**！

#### 🧭 方向常數元組清單：
我們可以直接將四個方向定義為由四個「位移元組」組成的清單：
```python
DIRECTIONS = [
    (-1, 0),  # 0: 向上 (列 - 1, 行不變)
    (1, 0),   # 1: 向下 (列 + 1, 行不變)
    (0, -1),  # 2: 向左 (列不變, 行 - 1)
    (0, 1)    # 3: 向右 (列不變, 行 + 1)
]
```
當我們想要探測目前座標 `curr = (r, c)` 的四個相鄰鄰居時，迴圈寫法變得極具幾何美感：
```python
for dr, dc in DIRECTIONS:
    nr = r + dr
    nc = c + dc
    if 0 <= nr < R and 0 <= nc < C:
        # 合法鄰居座標 (nr, nc)
```
每個方向向量都是不可變的常數元組，不用擔心被迴圈意外覆蓋，這正是高分競技程式中最推崇的標準工程寫法！


In [ ]:
# 範例 10.2.4：方向向量元組與四方向相鄰座標探測實戰

R, C = 5, 5  # 5x5 網格
start_pos = (2, 2)  # 中心出發點

# 定義四方向向量元組清單（上下左右）
DIRECTIONS = [
    (-1, 0),  # 上
    (1, 0),   # 下
    (0, -1),  # 左
    (0, 1)    # 右
]

r, c = start_pos
print(f"目前所在起點：({r}, {c})")

print()
print("--- 四方向相鄰合法鄰居座標 ---")
for dr, dc in DIRECTIONS:
    nr = r + dr
    nc = c + dc
    # 邊界守護
    if 0 <= nr < R and 0 <= nc < C:
        next_pos = (nr, nc)
        print(f"位移向量 ({dr:+2d}, {dc:+2d}) ➔ 到達鄰居：{next_pos}")


In [ ]:
# 填空 10.2.4：補齊八方向相鄰探測之左上與右下對角向量
# 請將 ___ 替換為正確的元組數值

# 八方向向量清單（上下左右 + 四個對角線）
DIRECTIONS_8 = [
    (-1, 0), (1, 0), (0, -1), (0, 1),
    (-1, -1),  # 左上
    (-1, 1),   # 右上
    (1, -1),   # 左下
    (___, ___)    # 右下：列加 1，行加 1
]

print("八方向向量總數：", len(DIRECTIONS_8))


In [ ]:
# 練習 10.2.4：網格安全位移探測器
# 題目說明：輸入兩個整數 R, C 代表網格尺寸。
# 接著輸入起點座標 r, c 與方向代碼 D（0:上, 1:下, 2:左, 3:右）。
# 請使用方向向量元組清單計算移動後的新座標 (nr, nc)。
# 輸出規則：
# 若新座標仍在網格內（0 <= nr < R 且 0 <= nc < C），輸出 "VALID nr nc"。
# 若新座標撞牆出界，輸出 "OUT OF BOUNDS"。

# 【公開測試資料 1】
# 4 4
# 0 1 0
# 輸出：
# OUT OF BOUNDS

# 【公開測試資料 2】
# 4 4
# 1 1 3
# 輸出：
# VALID 1 2

# 請在此處撰寫你的程式碼：
R, C = map(int, input().split())
r, c, d_code = map(int, input().split())

DIRECTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]

dr, dc = DIRECTIONS[d_code]
nr = r + dr
nc = c + dc

if 0 <= nr < R and 0 <= nc < C:
    print(f"VALID {nr} {nc}")
else:
    print("OUT OF BOUNDS")


In [ ]:
# 挑戰 10.2.4：連續向量步進路徑生成
# 題目說明：給定起點座標 (r, c)。
# 接著輸入一個整數 M 代表移動步數。
# 隨後有 M 行，每行輸入一個方向代碼 D（0:上, 1:下, 2:左, 3:右）。
# 請依序更新座標，並將過程中所踩過的「所有座標元組（包含起點）」加入一個串列中。
# 最後印出走過的所有座標（每行印一個元組）。
# 本題無公開測試資料，請自行測試。

# 請在此處撰寫你的程式碼：
r, c = map(int, input().split())
m = int(input())

DIRECTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
path = [(r, c)]

for _ in range(m):
    d = int(input())
    dr, dc = DIRECTIONS[d]
    r += dr
    c += dc
    path.append((r, c))

for p in path:
    print(p)


### 10.2.5 多傳回值模擬（為第 11 章自訂函式多值 return 提早鋪墊）

在後續的第十一章中，我們將會學習「自訂函數（Function）」。在許多其他程式語言（如 C/C++ 或 Java）中，一個函數通常只能嚴格回傳「單一一個數值」；如果想同時回傳「最大值」與「最小值」，往往要大費周章宣告指標或自訂結構。

但在 Python 中，工程師可以無比瀟灑地同時回傳多個值：
```python
# 預告第十一章的寫法：
# return min_val, max_val, total
```
#### 🎁 語法糖背後的功臣：自動元組打包（Tuple Packing）
為什麼 Python 能做到這件事？
因為當你用逗號隔開多個變數並輸出或賦值時，Python 在底層會**自動將它們打包為一個元組**！
就算你連小括號都懶得打：
```python
result = 10, 20, 30  # 這行完全合法！
print(type(result))  # 輸出：<class 'tuple'>
```
在主程式接收時，又透過上一節學過的「元組解包」，順暢地分派給各個變數：
```python
a, b, c = result
```
在我們當前尚未學習 `def` 函數的階段，這種「**打包多項統計指標為單一元組匯總物件**」的思維，能讓我們在處理多維統計資料時，邏輯極度清晰、程式模組化層次分明！


In [ ]:
# 範例 10.2.5：多值統計打包與解包接收實戰

scores = [85, 92, 78, 96, 88]

# 在一個運算區塊中，同時計算出多個指標並打包為單一元組
# 包含：(最高分, 最低分, 總分, 平均分)
highest = max(scores)
lowest = min(scores)
total = sum(scores)
avg = total // len(scores)

# 自動元組打包（不用打括號也行，加上括號更具備可讀性）
summary_report = (highest, lowest, total, avg)

print("匯總報告元組：", summary_report)
print("報告型態：", type(summary_report))

# 接收端一鍵解包
top, bottom, s_tot, s_avg = summary_report
print(f"榜首分：{top}，門檻分：{bottom}，總合：{s_tot}，均分：{s_avg}")


In [ ]:
# 填空 10.2.5：將奇數與偶數總和打包為雙值元組
# 請將 ___ 替換為正確的變數

numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

odd_sum = sum(x for x in numbers if x % 2 == 1)
even_sum = sum(x for x in numbers if x % 2 == 0)

# 打包成雙值元組
sums_tuple = (___, ___)

print("奇偶和元組：", sums_tuple)


In [ ]:
# 練習 10.2.5：四合一數據分析打包器
# 題目說明：輸入一串整數。
# 請計算出該數列的四項指標：
# 1. 元素總個數（N）
# 2. 最大值（MAX）
# 3. 最小值（MIN）
# 4. 總和（SUM）
# 將這四個數值依序打包為一個元組 `result = (N, MAX, MIN, SUM)`。
# 接著解包這四個數值，並輸出成單行（以空白隔開）。

# 【公開測試資料 1】
# 10 20 30 40
# 輸出：
# 4 40 10 100

# 【公開測試資料 2】
# 5 5 5
# 輸出：
# 3 5 5 15

# 請在此處撰寫你的程式碼：
nums = list(map(int, input().split()))

# 計算指標並打包
stats = (len(nums), max(nums), min(nums), sum(nums))

# 解包輸出
cnt, mx, mn, sm = stats
print(cnt, mx, mn, sm)


In [ ]:
# 挑戰 10.2.5：正負數與零計數三元組
# 題目說明：輸入一串整數。
# 統計其中：正數個數、負數個數、零的個數。
# 將三個計數打包為元組 `(pos_cnt, neg_cnt, zero_cnt)`。
# 印出該元組內容。
# 本題無公開測試資料，請自行測試。

# 請在此處撰寫你的程式碼：
nums = list(map(int, input().split()))

pos = sum(1 for x in nums if x > 0)
neg = sum(1 for x in nums if x < 0)
zero = sum(1 for x in nums if x == 0)

res = (pos, neg, zero)
print(res)


### 10.2.6 元組比大小與字典序排序原則（多準則評比原型）

在第五章 5-1 節中，我們學過文字字串的比大小是依循「字典序（Lexicographical Order）」。那麼，如果拿「兩個元組」互相比大小，例如 `(a1, b1) < (a2, b2)`，Python 會怎麼判定勝負呢？

#### ⚖️ 元組比大小的黃金字典序原則：
元組比大小的邏輯非常清晰嚴謹，完全比照字典的查閱流程：
1. **先比第 0 個元素**：
   如果 `a1 < a2`，判定第一個元組較小，**立刻結束比對，完全不看後面的數字！**
   如果 `a1 > a2`，判定第一個元組較大，同樣立刻結束！
2. **第一項平手時（`a1 == a2`），才比第 1 個元素**：
   接著比對 `b1` 與 `b2`。依此類推，直到分出勝負為止。

#### 🏆 APCS 考場的制勝神器：多準則排序原型
這種特性在 APCS 考場上有著無比巨大的價值！
試想一個超經典考題：「**運動會選手排名**」：
- 首要條件：金牌數多的排前面。
- 次要條件：若金牌數相同，銀牌數多的排前面。
- 再次條件：若銀牌數依然相同，學號小的排前面！

在 Python 中，我們根本不需要寫複雜又容易 bug 的巢狀 if-else，只要為每位選手建構一個「**排序鍵值元組**」：
`(-gold, -silver, student_id)`
因為負金牌數由小到大排，就等於原金牌數由大到小排！Python 依據元組字典序比對，平手時自動往後遞補判定，一行就能實現極為高階的多條件排序！


In [ ]:
# 範例 10.2.6：元組比大小與多條件排序原型展示

# 1. 基礎元組比大小：由左至右逐項對比
t1 = (5, 100)
t2 = (8, 1)
# 雖然 100 遠大於 1，但因為第一項 5 < 8，所以 t1 < t2 成立！
print("(5, 100) < (8, 1) 結果：", t1 < t2)

# 2. 首項平手時，比對次項
t3 = (5, 20)
t4 = (5, 50)
print("(5, 20) < (5, 50) 結果：", t3 < t4)

# 3. APCS 實戰原型：多條件選手排名
# 選手資料：(姓名, 金牌數, 銀牌數, 學號)
athletes = [
    ("Alice", 2, 5, 101),
    ("Bob", 3, 1, 102),
    ("Charlie", 2, 5, 100),  # 與 Alice 金銀牌相同，但學號較小
    ("David", 2, 3, 104)
]

# 排序邏輯：負金牌 (多到少), 負銀牌 (多到少), 正學號 (小到大)
# 我們為每位選手包裝一個排序元組：((-gold, -silver, id), name)
ranked = []
for name, g, s, sid in athletes:
    rank_key = (-g, -s, sid)
    ranked.append((rank_key, name, g, s, sid))

ranked.sort()  # 利用元組自動字典序排序！

print()
print("--- 最終排名榜單 ---")
for rk, name, g, s, sid in ranked:
    print(f"選手：{name:<7} | 金牌：{g}，銀牌：{s}，學號：{sid}")


In [ ]:
# 填空 10.2.6：建構雙條件元組進行大小比對
# 請將 ___ 替換為正確的變數，完成排序鍵值元組的建構

# 題目：比較兩位考生，優先比分數（高者勝），平手時比扣分次數（少者勝）
p1_score, p1_penalties = 90, 2
p2_score, p2_penalties = 90, 1

# 鍵值：(分數負號取反使高分排前, 扣分正數使少扣分者排前)
key1 = (-p1_score, p1_penalties)
key2 = (-p2_score, ___)

# 鍵值較小者代表表現更優秀（因為負分數更小代表原分數更高，罰分更少代表更小）
p2_is_better = (key2 < key1)
print("選手 2 是否勝出：", p2_is_better)


In [ ]:
# 練習 10.2.6：兩位選手多條件勝負判定
# 題目說明：輸入兩行，分別代表選手 A 與選手 B 的競賽數據。
# 每行三個整數：解題數（Solved）、總罰時（Penalty）、選手編號（ID）。
# 優勝規則：
# 1. 解題數較多者優勝。
# 2. 若解題數相同，總罰時較少者優勝。
# 3. 若依然相同，選手編號較小者優勝。
# 請將兩位選手的數據包裝為排序元組（利用負號進行方向切換），判定勝者並輸出該選手的 ID。

# 【公開測試資料 1】
# 4 120 1
# 5 300 2
# 輸出：
# 2

# 【公開測試資料 2】
# 4 150 10
# 4 120 20
# 輸出：
# 20

# 請在此處撰寫你的程式碼：
s1, p1, id1 = map(int, input().split())
s2, p2, id2 = map(int, input().split())

# 建構比較鍵值：(-solved, penalty, id)
key1 = (-s1, p1, id1)
key2 = (-s2, p2, id2)

if key1 < key2:
    print(id1)
else:
    print(id2)


In [ ]:
# 挑戰 10.2.6：三點距離原點多準則排序
# 題目說明：輸入三個座標點 r, c。
# 排序規則：
# 1. 距離原點曼哈頓距離（|r| + |c|）較小者排前面。
# 2. 若距離相同，r 座標較小者排前面。
# 3. 若 r 依然相同，c 座標較小者排前面。
# 請輸出排序後的第一名座標（以 r c 格式輸出）。
# 本題無公開測試資料，請自行測試。

# 請在此處撰寫你的程式碼：
pts = []
for _ in range(3):
    r, c = map(int, input().split())
    dist = abs(r) + abs(c)
    # 封裝元組：(距離, r, c)
    pts.append((dist, r, c))

pts.sort()
best = pts[0]
print(best[1], best[2])


## 本單元重點回顧與核心心法

在單元 10-2 中，我們從記憶體底層與演算法應用視角，全面解鎖了元組的高階威力：

1. **不可變性的記憶體本質**：
   - 串列修改時其 `id()` 門牌號碼保持不變，而元組內容一旦變更都是在記憶體中建立全新實體。
2. **可雜湊性（Hashability）指紋機制**：
   - 只有內容終生不變的不可變物件才能通過 `hash()` 計算出唯一指紋，這是元組能夠作為 `set` 元素與 `dict` 鍵值的底層鑰匙！
3. **巢狀元組結構化打包**：
   - 元組內嵌套元組，免類別物件即能實現多層級資料組織（如 `(id, name, (scores))`），支援多層索引與多層解包。
4. **二維向量位移模型**：
   - 將方向差值封裝為常數元組清單 `[(-1, 0), (1, 0), ...]`，大幅精簡網格走訪程式碼，兼具優雅與安全性。
5. **多傳回值與自動元組打包**：
   - 逗號隔開的多項數據會自動打包為元組，為跨區塊資料傳遞與多指標統計提供極致便利。
6. **元組字典序與多準則排序原型**：
   - 依序由左至右逐項對比，巧妙搭配負號（`-score`），一行代碼輕鬆實現 APCS 競賽中最繁瑣的多條件優先級排名！

---
🏆 **通關恭喜**！掌握了元組的不可變性與雜湊基礎後，你已經拿到了通往雜湊世界的黃金鑰匙！下一單元（10-3），我們將正式踏入 Python 最具殺傷力的王牌容器——**字典（Dictionary）**！
